In [ ]:
# Run in conda activate seuratenv
library(anndata)
library(Seurat)
library(Signac)
library(dplyr)

In [ ]:
mtxGene = read_h5ad(filename = "/group/ll001/dmakosa/projectdata/marina_snATAC/h5ad/combined_4.geneMtx.h5ad")
mtxPeak = read_h5ad(filename = "/group/ll001/dmakosa/projectdata/marina_snATAC/h5ad/combined_4.peak_matrix.h5ad")
data2kb = CreateSeuratObject(counts = t(as.matrix(mtxGene$X)), meta.data = mtxGene$obs,min.features = 0, min.cells = 0, assay = "ACTIVITY")
dataPeak = CreateSeuratObject(counts = t(as.matrix(mtxPeak$X)), meta.data = mtxPeak$obs,min.features = 0, min.cells = 0, assay = "PEAKS")
saveRDS(data2kb,"/group/ll001/dmakosa/projectdata/marina_snATAC/h5ad/combined_4.geneMtx.rds")
saveRDS(dataPeak,"/group/ll001/dmakosa/projectdata/marina_snATAC/h5ad/combined_4.peak_matrix.rds")

In [ ]:
peakMtx_atac = readRDS("/group/ll001/dmakosa/projectdata/marina_snATAC//h5ad/combined_4.peak_matrix.rds")

In [ ]:
peakMtx_atac <- RunTFIDF(peakMtx_atac)

In [ ]:
peakMtx_atac <- FindTopFeatures(peakMtx_atac, min.cutoff = 'q0')

In [ ]:
peakMtx_atac <- RunSVD(peakMtx_atac)

In [ ]:
DepthCor(peakMtx_atac)

In [ ]:
peakMtx_atac <- RunUMAP(object = peakMtx_atac, reduction = 'lsi', dims = 2:30)

In [ ]:
peakMtx_atac <- FindNeighbors(object = peakMtx_atac, reduction = 'lsi', dims = 2:30)

In [ ]:
peakMtx_atac <- FindClusters(object = peakMtx_atac, verbose = FALSE, algorithm = 3)

In [ ]:
DimPlot(object = peakMtx_atac, label = TRUE) + NoLegend()

In [ ]:
peakMtx_atac@reductions$lsi

In [ ]:
geneMtx2kb_atac = readRDS("/group/ll001/dmakosa/projectdata/marina_snATAC//h5ad/combined_4.geneMtx.rds")

In [ ]:
peakMtx_atac[['ACTIVITY']] = geneMtx2kb_atac[['ACTIVITY']]

In [ ]:
peakMtx_atac = NormalizeData(object = peakMtx_atac, assay = 'ACTIVITY', normalization.method = 'LogNormalize', scale.factor = median(peakMtx_atac$nCount_ACTIVITY))

## SAVE THE OBJECT PRIOR TO INTEGRATION

In [ ]:
saveRDS(peakMtx_atac, file = "/group/ll001/dmakosa/projectdata/marina_snATAC/h5ad/processedATAC.beforeIntegration.RemoveFirstComponent.rds")

## Now perform the integration using the ACTIVITY or ACTIVITY_2kb assay

In [85]:
# Read in scRNA-seq atlas
marina_rna <- readRDS("/group/llshared/rambo/integrated_col_trajectories.rds")

In [ ]:
marina_rna

In [88]:
DefaultAssay(peakMtx_atac) <- 'ACTIVITY'

In [ ]:
transfer.anchors <- FindTransferAnchors(
  reference = marina_rna,
  query = peakMtx_atac,
  reduction = 'cca',
  dims = 1:30
)

In [ ]:
# This one is with removing first component, consistent as the snATAC-seq UMAP that I generated beforehand
predicted.labels <- TransferData(
  anchorset = transfer.anchors,
  refdata = marina_rna$identity,
  weight.reduction = peakMtx_atac[['lsi']],
  dims = 2:30
)

In [ ]:

peakMtx_atac <- AddMetaData(object = peakMtx_atac, metadata = predicted.labels)

In [ ]:
plot1 <- DimPlot(marina_rna, group.by = 'identity', label = TRUE, repel = TRUE) + NoLegend() + ggtitle('scRNA-seq')
plot2 <- DimPlot(peakMtx_atac, group.by = 'predicted.id', label = TRUE, repel = TRUE) + NoLegend() + ggtitle('scATAC-seq')
plot1 + plot2

In [ ]:
saveRDS(peakMtx_atac, file = "/group/ll001/dmakosa/projectdata/marina_snATAC/h5ad/processedATAC.afterIntegration.RemoveFirstComponent.rds")